# Exercise 9b — Cofolding (optional)

Docking asks *where does this ligand sit in this structure*. **Cofolding** asks something harder:
*given only a protein sequence and a ligand, what does the complex look like?* No crystal
structure, no binding site, no search box. Boltz-2 also predicts an affinity while it is at it.

This notebook runs one cofolding prediction and packages the result so that
**exercise 9 can read it**. Everything here is optional — exercise 9 falls back to a prepared
example if you skip it.

**Before you start: Runtime → Change runtime type → T4 GPU.** Without a GPU this will not run.

How long it takes: about **ten minutes**, nearly all of it installing software and downloading
model weights. The prediction itself took **five minutes** when we timed it.

Adapted from kimjc95's [`Boltz_on_Colab.ipynb`](https://colab.research.google.com/github/kimjc95/computational-chemistry/blob/main/Boltz_on_Colab.ipynb)
(KAIST, BSD-3), which is where the dependency list comes from. Boltz-2 itself is
[jwohlwend/boltz](https://github.com/jwohlwend/boltz).

In [ ]:
# @title 1. Check you actually have a GPU { display-mode: "form" }
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader \
    || print("NO GPU — go to Runtime → Change runtime type → T4 GPU, then re-run this cell")
!python --version

## 2. Install

Three things here are not in the upstream instructions, and all three are needed. They are the
kind of breakage you meet whenever you run research code that is a year old:

* **`--ignore-requires-python`** — Boltz declares `requires-python = ">=3.10,<3.13"` and Colab is
  on 3.13. The cap exists because of pins (`numpy<2.0`, `scipy==1.13.1`) that this install never
  applies anyway: the dependencies go in unpinned *first*, then Boltz goes in with `--no-deps`.
* **the `sed`** — Boltz asks for `bf16-mixed` precision, which a T4 does not support.
* **the `run_boltz.py` wrapper** — PyTorch 2.6 made `torch.load` refuse anything but plain
  tensors, and the Boltz checkpoint carries a config object. Boltz has no upper pin on torch, so
  pip installs a version that refuses Boltz's own weights.

Run it and go and get a coffee.

In [ ]:
# @title 2. Install Boltz-2  (several minutes) { display-mode: "form" }
import time, pathlib
_t0 = time.time()

DEPS = ("numpy hydra-core pytorch-lightning rdkit dm-tree requests pandas types-requests "
        "einops einx fairscale mashumaro modelcif wandb click pyyaml biopython scipy numba "
        "gemmi scikit-learn chembl_structure_pipeline "
        "cuequivariance_ops_cu12 cuequivariance_ops_torch_cu12 cuequivariance_torch")

!pip install -q torch torchvision torchaudio
!git clone -q https://github.com/jwohlwend/boltz.git /content/boltz
!sed -i 's/bf16-mixed/32-true/g' /content/boltz/src/boltz/main.py
!grep -q "32-true" /content/boltz/src/boltz/main.py && echo "precision patch applied" \
    || echo "!! precision patch did NOT apply — the prediction will fail on a T4"
!pip install -q {DEPS}
!cd /content/boltz && pip install -q --no-deps --ignore-requires-python -e .

pathlib.Path("/content/run_boltz.py").write_text('''import torch
_real_load = torch.load
def _load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _real_load(*args, **kwargs)
torch.load = _load
from boltz.main import cli
cli()
''')

!boltz --help | head -3
print(f"\n>>> install took {time.time() - _t0:.0f} s")

## 3. What to fold

Two inputs: a protein **sequence** and a ligand **SMILES**. That is the whole point — no
structure goes in.

The defaults are the same system as exercise 9, so the two are directly comparable: the EGFR
kinase domain (residues 696–1022 of UniProt P00533) and gefitinib.

**To use your own compound,** replace `SMILES` — and if you want to compare against exercise 9's
redocking control, use the compound you docked. **To fold the resistant mutant instead,** set
`MUTATIONS` to `{790: "M", 858: "R"}`. **For a different protein,** paste its sequence and set
`FIRST_RESIDUE` to the residue number its first letter corresponds to; exercise 9 needs that
number to line the prediction up with a crystal structure.

In [ ]:
# EGFR kinase domain, residues 696-1022 of UniProt P00533
SEQUENCE = ("GEAPNQALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNP"
            "HVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQ"
            "HVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSI"
            "LEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMD"
            "EEDMDDVVDADEYLIPQQG")
FIRST_RESIDUE = 696      # the residue number of SEQUENCE[0], in the PDB's numbering
SMILES        = "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1"   # gefitinib
MUTATIONS     = {}       # e.g. {790: "M", 858: "R"} for the resistant T790M/L858R form
JOB           = "gefitinib_egfr_wt"   # names every output file — rename it if you change
                                      # anything above, or you will not know which is which

import pathlib
from rdkit import Chem

assert Chem.MolFromSmiles(SMILES) is not None, "that SMILES does not parse — check it"

# Apply the mutations by residue number, so you never have to count along the string yourself.
sequence = list(SEQUENCE)
for number, new in MUTATIONS.items():
    index = number - FIRST_RESIDUE
    assert 0 <= index < len(sequence), f"residue {number} is outside this sequence"
    print(f"residue {number}: {sequence[index]} -> {new}")
    sequence[index] = new
sequence = "".join(sequence)

pathlib.Path(f"/content/{JOB}.yaml").write_text(f'''version: 1
sequences:
  - protein:
      id: A
      sequence: {sequence}
  - ligand:
      id: B
      smiles: '{SMILES}'
properties:
  - affinity:
      binder: B
''')

print(f"\n{len(sequence)} residues, numbered {FIRST_RESIDUE}-"
      f"{FIRST_RESIDUE + len(sequence) - 1}")
print(f"ligand: {Chem.MolFromSmiles(SMILES).GetNumAtoms()} heavy atoms")
print(f"wrote /content/{JOB}.yaml")

## 4. Predict

`--use_msa_server` sends the sequence to the public ColabFold MMseqs2 server to build an
alignment — the model wants evolutionary information, not just the sequence. The first run also
downloads the model weights, so some of this time is one-off.

Structure sampling is turned down (`--sampling_steps 50`, one sample) to keep this inside a
practical session. Affinity sampling is left at its default, so that number is the real one.

In [ ]:
# @title 4. Predict  (the slow one) { display-mode: "form" }
import time
_t1 = time.time()

!python /content/run_boltz.py predict /content/{JOB}.yaml \
    --use_msa_server \
    --out_dir /content/boltz_out \
    --sampling_steps 50 \
    --diffusion_samples 1 \
    --output_format pdb

print(f"\n>>> prediction took {time.time() - _t1:.0f} s")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

## 5. Read it, and package it for exercise 9

`affinity_pred_value` is **log₁₀(IC₅₀ in µM)**, so it converts straight onto the pIC₅₀ axis the
rest of the course uses:

```
pIC50 = 6 - affinity_pred_value
```

Remember that pIC₅₀ is a *logarithmic* scale: one unit is a tenfold change in the concentration
you need. An error of 1 log unit means a factor of ten in potency.

The authors are explicit that this value is for **ranking active compounds against each other**,
not for absolute prediction — the same caveat a docking score carries.

The cell below writes a small `.zip` with the only three files exercise 9 reads, and downloads
it. Keep the two numbers it prints.

In [ ]:
import json, pathlib, shutil

out = pathlib.Path("/content/boltz_out")
model      = next(out.rglob("*_model_0.pdb"), None)
affinity   = next(out.rglob("affinity_*.json"), None)
confidence = next(out.rglob("confidence_*_model_0.json"), None)

if model is None:
    raise FileNotFoundError("no predicted structure found — did cell 4 finish? "
                            f"{sorted(p.name for p in out.rglob('*'))[:20]}")

if affinity:
    value = json.loads(affinity.read_text())["affinity_pred_value"]
    print(f"affinity_pred_value  {value:+.3f}   (= log10 IC50 in uM)")
    print(f"predicted pIC50      {6 - value:.2f}")
if confidence:
    c = json.loads(confidence.read_text())
    print(f"\nthe model's own confidence: ipTM {c['iptm']:.2f}  "
          f"(how sure it is about the interface)")
    print(f"                            pLDDT {c['complex_plddt']:.2f} "
          f"(how sure it is about the fold)")

bundle = pathlib.Path(f"/content/{JOB}_for_exercise9")
shutil.rmtree(bundle, ignore_errors=True)
bundle.mkdir()
for found in (model, affinity, confidence):
    if found:
        shutil.copy(found, bundle / found.name)
archive = shutil.make_archive(str(bundle), "zip", bundle)
print(f"\npackaged {len(list(bundle.iterdir()))} files into {archive} "
      f"({pathlib.Path(archive).stat().st_size / 1000:.0f} kB)")

print(f'''
--- to use this in exercise 9 -------------------------------------------
1. upload {pathlib.Path(archive).name} to exercise 9's Colab session
2. unzip it:      !unzip -o -q {pathlib.Path(archive).name} -d /content/boltz_out
3. in the settings cell, set:

       BOLTZ_RESULTS = "/content/boltz_out"
       BOLTZ_OFFSET  = {FIRST_RESIDUE - 1}
-------------------------------------------------------------------------''')

try:
    from google.colab import files
    files.download(archive)
except Exception as problem:
    print(f"\n(automatic download unavailable: {problem})")
    print("Download it yourself from the file browser in the left-hand sidebar.")

## What you have, and what it is worth

Exercise 9 will measure this prediction the same way it measured the docking: superpose the
predicted protein onto the crystal structure, then see how far the predicted ligand is from the
crystallographic one. Two things to watch for there.

**Read the backbone RMSD first.** If the predicted protein is wrong, a ligand disagreement may be
the protein's fault rather than the ligand's.

**And remember what the model has seen.** Gefitinib bound to EGFR has been in the PDB since 2004,
and Boltz-2 was trained on the PDB, so asking it to place gefitinib in EGFR is partly asking it to
recall. A fair test needs a complex deposited after its training cut-off — which is not something
you can arrange for your own compound this week. The honest move is to say so, rather than quote
the RMSD as a validation.